# Fairness in AI: Project 1 - Reina AL MASRI

## Introduction

This project analyzes a chest X-ray metadata dataset to identify and quantify potential demographic biases with respect to sensitive attributes (gender and age), and then apply a pre-processing mitigation technique.

*The dataset used is derived from the NIH Chest X-ray dataset, one of the largest publicly available collections of frontal-view chest radiographs. Each record contains patient-level metadata (age, gender, image dimensions), along with radiologist-assigned pathology labels. Because this dataset is widely used for training diagnostic models, any bias present in the data itself can impact directly model predictions, making fairness analysis an important first step before any machine learning.*

*Algorithmic fairness in healthcare is especially important: models that under-diagnose certain demographic groups can reinforce existing inequalities in medical care. Pre-processing mitigation techniques, which modify the training data before a model ever sees it, offer a way to address these biases at the source, without requiring changes to the model architecture itself. In this project, we apply and compare three such techniques: **reweighting** (as seen in the TDs), **massaging** (label flipping on borderline samples), and **uniform sampling** (balancing subgroup sizes through oversampling).*

*Here is my conjecture :*

- *Women are often neglected in healthcare (endometriosis for example)*
- *The more aged the person is, the higher chances they are sick.*

*However, we don't want a future model to say that a young person (with a sickness) is or is not sick just because of their age. That is also medical neglect. Same goes for women.*

*The analysis proceeds in several stages: first, we clean and preprocess the data (handling duplicates, extreme values, and creating the binary target); then, we explore the dataset and quantify biases using statistical tests and fairness metrics from aif360; finally, we apply three pre-processing mitigation strategies and compare their effectiveness at reducing bias across both gender and age.*

**Outline:**
1. Imports
2. Data Cleaning & Preprocessing
3. Dataset Overview & Pathology Analysis
4. Bias Analysis (Gender, Age, Crossed)
5. Fairness Metrics on Ground Truth
6. Bias Mitigation (Reweighting, Massaging, Sampling)
7. Conclusion

### 1. **Imports and `get_metrics` function**

In [62]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2_contingency, mannwhitneyu
from aif360.sklearn.metrics import *
from aif360.algorithms.preprocessing import Reweighing as AIF_Reweighing
from aif360.datasets import StandardDataset
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [4]:
# this is just the get_metrics function that was given in td1 and td2
def get_metrics(
    y_true, # list or np.array of truth values
    y_pred=None,  # list or np.array of predictions
    prot_attr=None, # list or np.array of protected/sensitive attribute values
    priv_group=1, # value taken by the privileged group
    pos_label=1, # value taken by the positive truth/prediction
    sample_weight=None # list or np.array of weights value,
):
    group_metrics = {}
    group_metrics["base_rate_truth"] = base_rate(
        y_true=y_true, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["statistical_parity_difference"] = statistical_parity_difference(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["disparate_impact_ratio"] = disparate_impact_ratio(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    if not y_pred is None:
        group_metrics["base_rate_preds"] = base_rate(
        y_true=y_pred, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["equal_opportunity_difference"] = equal_opportunity_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["average_odds_difference"] = average_odds_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        if len(set(y_pred))>1:
            group_metrics["conditional_demographic_disparity"] = conditional_demographic_disparity(
                y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
            )
        else:
            group_metrics["conditional_demographic_disparity"] =None
        group_metrics["smoothed_edf"] = smoothed_edf(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["df_bias_amplification"] = df_bias_amplification(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["balanced_accuracy_score"] = balanced_accuracy_score(
        y_true=y_true, y_pred=y_pred, sample_weight=sample_weight
        )
    return group_metrics


### 2. **Data Cleaning & Preprocessing**

Before any analysis, we need to clean the data. Let's take a look.

In [ ]:
df = pd.read_csv('AL_MASRI_REINA.csv')
print(f"Shape: {df.shape}")
df.head()

Shape: (54490, 11)


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168
3,00000005_000.png,No Finding,0,5,69,F,PA,2048,2500,0.168,0.168
4,00000005_001.png,No Finding,1,5,69,F,AP,2500,2048,0.168,0.168


**Step 1 : Checking for missing values**

In [8]:
# we should check for any NaN values (i'm assuming there's none)
# to know if we need to replace any cells (with the mean for example)

for column in df.columns:
  print(column, df[column].isnull().values.any())

Image Index False
Finding Labels False
Follow-up # False
Patient ID False
Patient Age False
Patient Gender False
View Position False
OriginalImage[Width False
Height] False
OriginalImagePixelSpacing[x False
y] False


We can see that we have no missing values.


**Step 2: Creating Binary Target**

*Reasoning: Since the 'Finding Labels' column is either a label or 'No Finding', and our goal is to predict if the person is sick or not, I decided to change it to a binary column (is or is not sick)*

In [9]:
df['has_finding'] = (df['Finding Labels'] != 'No Finding').astype(int)
label = 'has_finding'

**Step 3: Handling Extreme Values (Age > 100)**

*Reasoning: I actually proceeded to do the analysis of bias in the data before even handling extreme values. That's where I noticed there are people aged 155 for example. We also notice that patients are repeating in the dataset (they often have follow ups, or multiple images from different views, e.g., AP or PA). I used this to my advantage and searched for the patients with invalid age within the dataset, to see if there exists a valid age value.*

In [10]:
# 1. identifying extreme ages
extreme_age_count = (df['Patient Age'] > 100).sum()
print(f"Patients with age > 100: {extreme_age_count}")

Patients with age > 100: 6


In [11]:
extreme_ages = df[df['Patient Age'] > 100][['Patient ID', 'Patient Age', 'View Position', 'Finding Labels']].sort_values('Patient ID')
print("Extreme age records:")
print(extreme_ages)

Extreme age records:
       Patient ID  Patient Age View Position Finding Labels
23590       12238          148            PA     No Finding
27078       13950          148            PA     No Finding
35987       18366          152            PA   Pneumothorax
47724       26028          154            PA    Atelectasis
49141       26871          155            PA     No Finding
50909       27989          155            PA     No Finding


In [12]:
# 2. handle age > 100

# for each patient with age > 100, i'll check if they have other records
ind = [] # indices to remove

for id in df[df['Patient Age'] > 105]['Patient ID'].unique(): # .unique() because patients repeat
    data = df[df['Patient ID'] == id]
    
    # get other ages for this patient (if they exist)
    valid_ages = data[data['Patient Age'] <= 105]['Patient Age'].values
    
    if len(valid_ages) > 0:
        # patient has other records with valid age, we can use the median of valid ages
        age = int(np.median(valid_ages)) # this is the correct age
        print(f"Patient {id}: valid ages {valid_ages}, using {age}")
        df.loc[df['Patient ID'] == id, 'Patient Age'] = age # replace with the valid age
    else:
        # no valid age found, so i put it in the list of ages to remove
        print(f"Patient {id}: no valid age found, age : {df[df['Patient ID'] == id]['Patient Age']}")
        ind.extend(data.index.tolist())

Patient 12238: valid ages [63 63 63 63 63 63 64 63 63 64 64 64 64 64], using 63
Patient 13950: valid ages [64 64 64 64 65 65 65], using 64
Patient 18366: valid ages [64 65 64 64 64 64 64 65 64 64 65 64 64 64 65 65 65 64 64 64 64 64 64 64
 64 64 64 64 64 64 64 64 64 64 64 64 64 64 64 65 64 64 64 64 64 64 64 64
 64 64 64 64 65 65 65 65 66 66 66 66 66 66 66 66 66 66 66], using 64
Patient 26028: valid ages [60], using 60
Patient 26871: no valid age found, age : 49141    155
Name: Patient Age, dtype: int64
Patient 27989: no valid age found, age : 50909    155
Name: Patient Age, dtype: int64


*After this step, we see two people have invalid ages (155, not even close to 100). I decided to remove them completely from the dataset.*

In [13]:
# 3. remove rows with no valid age
df = df.drop(ind).reset_index(drop=True)
print(f"{df.shape}") # used to be 54490 rows × 11 columns

(54488, 12)


**Step 4 : Aggregating Patient Records**

In [15]:
patient_records = df.groupby('Patient ID').size().sort_values(ascending=False)
patient_records

Patient ID
10007    184
15530    158
13993    143
19124    130
11237    116
        ... 
13766      1
13768      1
13770      1
13771      1
30803      1
Length: 14998, dtype: int64

In [16]:
print(f"unique patients: {len(patient_records)}")
print(f"patients with multiple images: {(patient_records > 1).sum()}")

unique patients: 14998
patients with multiple images: 6489


In [17]:
# 1. if a patient has any finding in any image (AP or PA), they have finding
patient_findings = df.groupby('Patient ID')['has_finding'].max()  # = 1 if any image shows finding

# 2. update has_finding 
df['has_finding'] = df['Patient ID'].map(patient_findings).astype(int)

print(f"\nAfter aggregation by patient:")
print(df['has_finding'].value_counts())
print(f"Positive rate: {df['has_finding'].mean():.4f}")


After aggregation by patient:
has_finding
1    42972
0    11516
Name: count, dtype: int64
Positive rate: 0.7887


In [18]:
# 3. deduplicate
df = df.drop_duplicates(subset=['Patient ID'], keep='first').reset_index(drop=True)

print(df.shape)
print(df['has_finding'].value_counts())

(14998, 12)
has_finding
0    7976
1    7022
Name: count, dtype: int64


### 3. **Dataset Overview**

*Now that our data is clean and deduplicated (one row per patient new shape = 14998, 12), let's explore what we're working with.*

In [19]:
# quick summary of the cleaned dataset
print(f"Observations: {len(df)}")
print(f"Columns: {len(df.columns)}")
print(f"\nColumn types:")
print(df.dtypes)
print(f"\nPositive rate (has finding): {df['has_finding'].mean()*100:.1f}%")
print(f"Negative rate (no finding):  {(1-df['has_finding'].mean())*100:.1f}%")

Observations: 14998
Columns: 12

Column types:
Image Index                        str
Finding Labels                     str
Follow-up #                      int64
Patient ID                       int64
Patient Age                      int64
Patient Gender                     str
View Position                      str
OriginalImage[Width              int64
Height]                          int64
OriginalImagePixelSpacing[x    float64
y]                             float64
has_finding                      int64
dtype: object

Positive rate (has finding): 46.8%
Negative rate (no finding):  53.2%


In [20]:
# age & gender quick stats
print("Age statistics:")
print(df['Patient Age'].describe())
print(f"\nMean age (has finding):  {df[df['has_finding']==1]['Patient Age'].mean():.1f}")
print(f"Mean age (no finding):   {df[df['has_finding']==0]['Patient Age'].mean():.1f}")
print(f"\nGender split:")
print(df['Patient Gender'].value_counts())

Age statistics:
count    14998.000000
mean        46.084945
std         16.706735
min          1.000000
25%         34.000000
50%         48.000000
75%         58.000000
max         95.000000
Name: Patient Age, dtype: float64

Mean age (has finding):  49.2
Mean age (no finding):   43.4

Gender split:
Patient Gender
M    8021
F    6977
Name: count, dtype: int64


#### 3.1 Pathology Distribution

*The 'Finding Labels' column can contain multiple pathologies separated by `|`. Let's look at the individual pathologies and how they're distributed.*

In [21]:
# extract individual pathologies from the multi-label column
all_findings = []
for labels in df['Finding Labels']:
    if labels != 'No Finding':
        all_findings.extend(labels.split('|'))

unique_findings = sorted(set(all_findings))
print(f"Unique pathologies found: {len(unique_findings)}")
print(unique_findings)

# count how many patients have each pathology
finding_counts = pd.Series(all_findings).value_counts()
print(f"\nPathology frequencies:")
print(finding_counts)

Unique pathologies found: 14
['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']

Pathology frequencies:
Infiltration          1760
Nodule                 798
Atelectasis            793
Mass                   648
Effusion               608
Cardiomegaly           382
Pleural_Thickening     355
Fibrosis               291
Consolidation          192
Pneumothorax           132
Emphysema              130
Pneumonia               73
Edema                   41
Hernia                  38
Name: count, dtype: int64


In [22]:
# visualize pathology distribution
fig = px.bar(x=finding_counts.index, y=finding_counts.values,
             title='Frequency of Each Pathology (across all patients)',
             labels={'x': 'Pathology', 'y': 'Number of Occurrences'},
             text=finding_counts.values,
             color=finding_counts.values,
             color_continuous_scale='Tealrose')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_tickangle=-45, height=500, width=800, coloraxis_showscale=False)
fig.show()

*There's a big imbalance between pathologies. Some (like Infiltration) appear thousands of times, while others (like Hernia) are extremely rare. This is a form of label imbalance that could affect any model trained on this data.*

In [23]:
# how many pathologies does each patient have?
df['num_findings'] = df['Finding Labels'].apply(
    lambda x: 0 if x == 'No Finding' else len(x.split('|'))
)

num_dist = df['num_findings'].value_counts().sort_index()

fig = px.bar(x=num_dist.index, y=num_dist.values,
             title='Number of Pathologies per Patient',
             labels={'x': 'Number of Pathologies', 'y': 'Number of Patients'},
             text=num_dist.values,
             color_discrete_sequence=['#6BCB77'])
fig.update_traces(textposition='outside')
fig.update_layout(height=400, width=800)
fig.show()

print(f"Average number of pathologies per patient: {df['num_findings'].mean():.2f}")
print(f"Max pathologies for a single patient: {df['num_findings'].max()}")

Average number of pathologies per patient: 0.42
Max pathologies for a single patient: 6


*Most patients with findings have only 1 pathology, but some have 2 or more. This makes classification harder and could introduce compound biases (a patient with 3 pathologies is very different from one with 1)*

### 4. **Bias Analysis**

#### **4.1 Gender Distribution**

*We'll analyze the distribution of gender and its relationship with findings. We want to understand if there's any inherent bias in the data.*

In [24]:
# create binary gender (male = 1 as privileged, female = 0 as unprivileged)
df['gender'] = (df['Patient Gender'] == 'M').astype(int)

print(f"Males:   {(df['gender']==1).sum()} ({df['gender'].mean()*100:.1f}%)")
print(f"Females: {(df['gender']==0).sum()} ({(1-df['gender'].mean())*100:.1f}%)")
print(f"Ratio M/F: {(df['gender']==1).sum() / (df['gender']==0).sum():.2f}")

Males:   8021 (53.5%)
Females: 6977 (46.5%)
Ratio M/F: 1.15


In [25]:
# gender distribution: pie + bar side by side
fig = make_subplots(rows=1, cols=2, specs=[[{'type':'pie'}, {'type':'bar'}]],
                    subplot_titles=('Gender Proportions', 'Gender Counts'))

counts = df['Patient Gender'].value_counts()
fig.add_trace(go.Pie(labels=['Male', 'Female'], values=[counts['M'], counts['F']],
                     marker_colors=['#4A90E2', '#FF6B9D']), row=1, col=1)
fig.add_trace(go.Bar(x=['Male', 'Female'], y=[counts['M'], counts['F']],
                     marker_color=['#4A90E2', '#FF6B9D'],
                     text=[counts['M'], counts['F']], textposition='auto'), row=1, col=2)
fig.update_layout(height=400, width=800, showlegend=False, title_text='Gender Distribution in Dataset')
fig.show()

*The dataset is slightly male-dominated.*

**Finding Rate by Gender**

*Now let's see if there are differences in the rate of findings between males and females.*

In [26]:
# finding rate by gender
gender_stats = df.groupby('gender')['has_finding'].agg(['sum', 'count', 'mean'])
gender_stats.index = ['Female', 'Male']
gender_stats.columns = ['With Finding', 'Total', 'Finding Rate']
gender_stats['Finding Rate'] = (gender_stats['Finding Rate'] * 100).round(2).astype(str) + '%'
print(gender_stats)

        With Finding  Total Finding Rate
Female          3215   6977       46.08%
Male            3807   8021       47.46%


In [27]:
# visualization beause everything is better when it is visualized ! 

# most of the code is for 'visual' purposes only
# findings by gender: grouped bar (count) + percentage stacked
fig = make_subplots(rows=1, cols=2, subplot_titles=('Count by Gender', 'Percentage by Gender'))

for finding_val, label, color in [(0, 'No Finding', '#4ECDC4'), (1, 'Has Finding', '#FF6B6B')]:
    for gender_val, gender_name in [(0, 'Female'), (1, 'Male')]:
        count = ((df['gender']==gender_val) & (df['has_finding']==finding_val)).sum()
        fig.add_trace(go.Bar(x=[gender_name], y=[count], name=label,
                             marker_color=color, showlegend=(gender_val==0),
                             text=[count], textposition='auto'),
                      row=1, col=1)

# percentage view (same code but dividing by the total then *100)
for finding_val, label, color in [(0, 'No Finding', '#4ECDC4'), (1, 'Has Finding', '#FF6B6B')]:
    pcts = []
    for gender_val, gender_name in [(0, 'Female'), (1, 'Male')]:
        total = (df['gender']==gender_val).sum()
        pct = ((df['gender']==gender_val) & (df['has_finding']==finding_val)).sum() / total * 100
        pcts.append(pct)
    fig.add_trace(go.Bar(x=['Female', 'Male'], y=pcts, name=label,
                         marker_color=color, showlegend=False,
                         text=[f'{p:.1f}%' for p in pcts], textposition='auto'),
                  row=1, col=2)

fig.update_layout(height=400, width=800, barmode='group', title_text='Findings Distribution by Gender')
fig.show()

*The finding rates are very close between males and females (both around 55-57%), confirming that gender alone does not strongly predict the presence of a pathology in this dataset.*

In [34]:
for metric, value in gender_truth_metrics.items():
    if value is not None:
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: None")

base_rate_truth: 0.4682
statistical_parity_difference: -0.0138
disparate_impact_ratio: 0.9709
base_rate_preds: 0.4682
equal_opportunity_difference: 0.0000
average_odds_difference: 0.0000
conditional_demographic_disparity: -0.0010
smoothed_edf: 0.0296
df_bias_amplification: 0.0000
balanced_accuracy_score: 1.0000


#### Gender Bias Interpretation:

- *Disparate Impact Ratio (=0.97)*: *very close to perfect fairness (1.0).* *(it means females receive findings at ~97% the rate of male)*
- *Statistical Parity Difference (=0.01)*: *it means only ~1.4% difference in finding rates (slightly lower for females).*

This dataset shows fairly gender-balanced finding rates. The small difference could be due to natural medical variation.

*(Gender fairness looks good in the data but we should still monitor it after mitigation and during model training.)*

#### **4.2 Age Distribution & Bias Analysis**

*Age is particularly interesting because medical conditions naturally correlate with age, but we need to make sure future models don't discriminate unfairly.*

In [35]:
# create age groups for analysis (and plots)
df['age_group'] = pd.cut(df['Patient Age'], 
                         bins=[0, 30, 50, 70, 100], 
                         labels=['<30', '30-50', '50-70', '70+'])

print("Age group distribution:")
print(df['age_group'].value_counts().sort_index())

print("\nFinding rate by age group:")
age_group_stats = df.groupby('age_group')['has_finding'].agg(['sum', 'count', 'mean'])
age_group_stats.columns = ['Total with Finding', 'Total Count', 'Finding Rate']
print(age_group_stats)

Age group distribution:
age_group
<30      2941
30-50    5504
50-70    5664
70+       889
Name: count, dtype: int64

Finding rate by age group:
           Total with Finding  Total Count  Finding Rate
age_group                                               
<30                      1061         2941      0.360762
30-50                    2277         5504      0.413699
50-70                    3138         5664      0.554025
70+                       546          889      0.614173


In [36]:
# continuous age distribution + boxplot by finding
fig = make_subplots(rows=1, cols=2, subplot_titles=('Age Distribution (continuous)', 'Age by Finding Status'))

fig.add_trace(go.Histogram(x=df['Patient Age'], nbinsx=40, marker_color='#95E1D3', name='Age'), row=1, col=1)

for val, label, color in [(0, 'No Finding', '#4ECDC4'), (1, 'Has Finding', '#FF6B6B')]:
    fig.add_trace(go.Box(y=df[df['has_finding']==val]['Patient Age'], name=label, marker_color=color), row=1, col=2)

fig.update_layout(height=400, width=800, showlegend=True, title_text='Age Distribution Overview')
fig.show()

In [37]:
# findings by age group: count + percentage
fig = make_subplots(rows=1, cols=2, subplot_titles=('Count by Age Group', 'Percentage by Age Group'))

age_groups = ['<30', '30-50', '50-70', '70+']
age_colors = ['#FFD93D', '#6BCB77', '#4D96FF', '#FF6B6B']

for finding_val, label, color in [(0, 'No Finding', '#4ECDC4'), (1, 'Has Finding', '#FF6B6B')]:
    counts = [((df['age_group']==ag) & (df['has_finding']==finding_val)).sum() for ag in age_groups]
    fig.add_trace(go.Bar(x=age_groups, y=counts, name=label, marker_color=color,
                         text=counts, textposition='auto'), row=1, col=1)

for finding_val, label, color in [(0, 'No Finding', '#4ECDC4'), (1, 'Has Finding', '#FF6B6B')]:
    pcts = []
    for ag in age_groups:
        total = (df['age_group']==ag).sum()
        pct = ((df['age_group']==ag) & (df['has_finding']==finding_val)).sum() / total * 100 if total > 0 else 0
        pcts.append(pct)
    fig.add_trace(go.Bar(x=age_groups, y=pcts, name=label, marker_color=color,
                         showlegend=False, text=[f'{p:.1f}%' for p in pcts], textposition='auto'), row=1, col=2)

fig.update_layout(height=400, width=800, barmode='group', title_text='Findings Distribution by Age Group')
fig.show()

*The finding rate increases with age. The 50-70 and 70+ groups have higher proportions of findings. This is medically expected but this kind of correlation could cause an unfair classifier.*

**Binary Age for Fairness Metrics**

*For fairness metrics, we need a binary sensitive attribute. Let's split: young (<50) vs older (≥50).*

In [38]:
# creating binary age variable (>=50 = 1 as privileged/older, <50 = 0 as unprivileged/younger)
df['age_binary'] = (df['Patient Age'] >= 50).astype(int)

# some statistics
print("Binary age distribution:")
print(df['age_binary'].value_counts())
print(f"\nPercentage of older patients (≥50): {df['age_binary'].mean()*100:.2f}%")
print(f"Percentage of younger patients (<50): {(1-df['age_binary'].mean())*100:.2f}%")

print("\nFinding rate by age group:")
print(f"Young (<50) finding rate: {df[df['age_binary']==0]['has_finding'].mean():.4f}")
print(f"Older (≥50) finding rate: {df[df['age_binary']==1]['has_finding'].mean():.4f}")
print(f"Difference (Older - Young): {df[df['age_binary']==1]['has_finding'].mean() - df[df['age_binary']==0]['has_finding'].mean():.4f}")

Binary age distribution:
age_binary
0    8105
1    6893
Name: count, dtype: int64

Percentage of older patients (≥50): 45.96%
Percentage of younger patients (<50): 54.04%

Finding rate by age group:
Young (<50) finding rate: 0.3904
Older (≥50) finding rate: 0.5597
Difference (Older - Young): 0.1693


*The split is relatively balanced (~50/50), but the finding rate gap between young and older patients is very big (~17 percentage points).*

**Fairness Metrics on Ground Truth (Age)**

*Similar to gender, let's compute fairness metrics for age to understand inherent bias.*

#### **Age Bias Interpretation:**

- **Disparate Impact Ratio (=0.70)**: *Below the 0.8 fairness threshold. (it menas younger patients have findings at only 70% the rate of older patients.)*
- **Statistical Parity Difference (=-0.17)**: *17% gap in finding rates, younger patients have lower rates.*

*This is medically expected (conditions increase with age), but it means that a model trained on this data could potentially underdiagnose younger patients.*

#### **4.3 Crossed Analysis: Gender × Age**

*Let's check if gender and age distributions are independent, or if there are compound biases (for example, fewer young women than young men).*

In [41]:
# age by gender: boxplot + statistical test
stat, p = mannwhitneyu(
    df[df['Patient Gender']=='M']['Patient Age'],
    df[df['Patient Gender']=='F']['Patient Age']
)

fig = make_subplots(rows=1, cols=2, subplot_titles=('Age Distribution by Gender', 'Gender × Age Group'))

for gender, color in [('M', '#4A90E2'), ('F', '#FF6B9D')]:
    fig.add_trace(go.Box(y=df[df['Patient Gender']==gender]['Patient Age'],
                         name=gender, marker_color=color), row=1, col=1)

# gender × age group
cross = pd.crosstab(df['age_group'], df['Patient Gender'])
for gender, color in [('M', '#4A90E2'), ('F', '#FF6B9D')]:
    fig.add_trace(go.Bar(x=cross.index.astype(str), y=cross[gender], name=gender,
                         marker_color=color), row=1, col=2)

fig.update_layout(height=400, width=800, title_text='Gender × Age Interaction')
fig.show()

print(f"Mann-Whitney U test (age by gender): p-value = {p:.4e}")
print("Significant difference" if p < 0.05 else "No significant difference")

Mann-Whitney U test (age by gender): p-value = 1.1545e-11
Significant difference


*The Mann-Whitney test tells us whether the age distributions differ significantly between genders. They do in this case, this is a compound bias: gender and age are correlated in our dataset, which means any age-based unfairness could also affect genders unequally.*

#### 4.4 Crossed Analysis: Gender × Pathology

*Do certain pathologies affect one gender more than the other?.*

In [43]:
# chi-squared test: gender vs has_finding
contingency = pd.crosstab(df['Patient Gender'], df['has_finding'])
chi2, p_gender_finding, _, _ = chi2_contingency(contingency)

print(f"Chi-squared test (Gender × Finding): χ² = {chi2:.2f}, p = {p_gender_finding:.4e}")
print("Gender and finding are not independent (bias!)" if p_gender_finding < 0.05
      else "Gender and finding appear independent (no significant bias)")


Chi-squared test (Gender × Finding): χ² = 2.81, p = 9.3632e-02
Gender and finding appear independent (no significant bias)


*The chi-squared test tells us whether gender and having a finding are statistically independent. Let's also look at how individual pathologies differ between genders.*

In [44]:
# prevalence of each pathology by gender
rows = []
for finding in unique_findings:
    rate_m = df[df['Patient Gender']=='M']['Finding Labels'].apply(lambda x: finding in x).mean() * 100
    rate_f = df[df['Patient Gender']=='F']['Finding Labels'].apply(lambda x: finding in x).mean() * 100
    rows.append({'Pathology': finding, 'Male (%)': rate_m, 'Female (%)': rate_f,
                 'Diff (pp)': abs(rate_m - rate_f)})

gender_path_df = pd.DataFrame(rows).sort_values('Diff (pp)', ascending=False)
print("\nPrevalence by gender (sorted by difference):")
print(gender_path_df.to_string(index=False))


Prevalence by gender (sorted by difference):
         Pathology  Male (%)  Female (%)  Diff (pp)
       Atelectasis  5.859619    4.629497   1.230122
      Cardiomegaly  2.032166    3.138885   1.106719
              Mass  4.712629    3.869858   0.842771
      Pneumothorax  0.623364    1.175290   0.551927
         Emphysema  1.047251    0.659309   0.387942
            Nodule  5.460666    5.159811   0.300855
      Infiltration 11.868844   11.580909   0.287936
          Effusion  3.964593    4.156514   0.191921
         Pneumonia  0.573495    0.386986   0.186509
Pleural_Thickening  2.443586    2.278916   0.164669
     Consolidation  1.221793    1.347284   0.125491
            Hernia  0.199476    0.315322   0.115845
             Edema  0.261813    0.286656   0.024843
          Fibrosis  1.944895    1.934929   0.009966


*While some individual pathologies (e.g., Hernia, Cardiomegaly) show notable prevalence differences between genders, the overall binary finding rate remains similar. Let's visualize the top differences.*

In [45]:
# visualize top pathologies by gender
top_n = 10
top_pathologies = gender_path_df.head(top_n)

fig = go.Figure()
fig.add_trace(go.Bar(x=top_pathologies['Pathology'], y=top_pathologies['Male (%)'],
                     name='Male', marker_color='#4A90E2'))
fig.add_trace(go.Bar(x=top_pathologies['Pathology'], y=top_pathologies['Female (%)'],
                     name='Female', marker_color='#FF6B9D'))
fig.update_layout(barmode='group', title=f'Top {top_n} Pathologies with Largest Gender Difference',
                  xaxis_tickangle=-45, height=500, width=800,
                  yaxis_title='Prevalence (%)', xaxis_title='Pathology')
fig.show()

*Some pathologies show notable gender differences. But the chi squared test confirms there is no significant bias*

#### 4.5 Crossed Analysis: Age × Pathology

*Do certain pathologies vary significantly across age groups? And is there a statistical link between age and having a finding?*

In [49]:
# chi-squared test: age_group vs has_finding
contingency_age = pd.crosstab(df['age_group'], df['has_finding'])
chi2_age, p_age_finding, _, _ = chi2_contingency(contingency_age)

print(f"Chi-squared test (Age Group × Finding): χ² = {chi2_age:.2f}, p = {p_age_finding:.4e}")
print("Age and finding are not independent (bias!)" if p_age_finding < 0.05
      else "Age and finding appear independent")

# average number of findings by age group
avg_by_age = df.groupby('age_group')['num_findings'].mean()
print(f"\nAverage number of pathologies per patient by age group:")
print(avg_by_age)

Chi-squared test (Age Group × Finding): χ² = 445.65, p = 2.8614e-96
Age and finding are not independent (bias!)

Average number of pathologies per patient by age group:
age_group
<30      0.287657
30-50    0.346294
50-70    0.513771
70+      0.651294
Name: num_findings, dtype: float64


In [50]:
# prevalence of top pathologies by age group
top_5 = finding_counts.head(5).index.tolist()
age_groups_list = ['<30', '30-50', '50-70', '70+']
age_palette = ['#FFD93D', '#6BCB77', '#4D96FF', '#FF6B6B']

fig = go.Figure()
for ag, color in zip(age_groups_list, age_palette):
    mask = df['age_group'] == ag
    rates = [df[mask]['Finding Labels'].apply(lambda x, f=f: f in x).mean() * 100 for f in top_5]
    fig.add_trace(go.Bar(x=top_5, y=rates, name=ag, marker_color=color))

fig.update_layout(barmode='group', title='Top 5 Pathology Prevalence by Age Group',
                  yaxis_title='Prevalence (%)', xaxis_title='Pathology',
                  xaxis_tickangle=-45, height=500, width=800)
fig.show()

*As expected, older age groups have higher pathology prevalence for most conditions.*

### 5. **Bias Summary**

*Let's summarize all the biases we've found in a single table.*

In [ ]:
# summary of all biases found
gender_counts_vals = df['Patient Gender'].value_counts()
age_group_counts = df['age_group'].value_counts()

# build a summary table: each row = one type of bias, with its metric, severity level, and a note
bias_summary = pd.DataFrame([
    {'Bias Type': 'Gender Imbalance',
     'Metric': f"Ratio M/F = {gender_counts_vals['M']/gender_counts_vals['F']:.2f}",
     'Severity': 'LOW' if gender_counts_vals.max()/gender_counts_vals.min() < 1.5 else 'MODERATE',
     'Note': f"Males: {gender_counts_vals['M']/len(df)*100:.1f}%, Females: {gender_counts_vals['F']/len(df)*100:.1f}%"},
    {'Bias Type': 'Age Group Imbalance',
     'Metric': f"Ratio max/min = {age_group_counts.max()/age_group_counts.min():.2f}",
     'Severity': 'HIGH' if age_group_counts.max()/age_group_counts.min() > 3 else 'MODERATE',
     'Note': f"Largest: {age_group_counts.idxmax()}, Smallest: {age_group_counts.idxmin()}"},
    {'Bias Type': 'Pathology Imbalance',
     'Metric': f"Ratio max/min = {finding_counts.max()/finding_counts.min():.1f}",
     'Severity': 'HIGH',
     'Note': f"Most common: {finding_counts.index[0]}, Rarest: {finding_counts.index[-1]}"},
    {'Bias Type': 'Gender × Finding',
     'Metric': f"χ² p = {p_gender_finding:.2e}",
     'Severity': 'LOW' if p_gender_finding >= 0.05 else 'MODERATE',
     'Note': 'Significant' if p_gender_finding < 0.05 else 'Not significant'},
    {'Bias Type': 'Age × Finding',
     'Metric': f"χ² p = {p_age_finding:.2e}",
     'Severity': 'HIGH' if p_age_finding < 0.001 else 'MODERATE',
     'Note': 'Strong association between age and pathology presence'},
])

print(bias_summary.to_string(index=False))

          Bias Type               Metric Severity                                                  Note
   Gender Imbalance     Ratio M/F = 1.15      LOW                          Males: 53.5%, Females: 46.5%
Age Group Imbalance Ratio max/min = 6.37     HIGH                         Largest: 50-70, Smallest: 70+
Pathology Imbalance Ratio max/min = 46.3     HIGH             Most common: Infiltration, Rarest: Hernia
   Gender × Finding      χ² p = 9.36e-02      LOW                                       Not significant
      Age × Finding      χ² p = 2.86e-96     HIGH Strong association between age and pathology presence


In [52]:
# visualize bias summary (so it's alot easier to read the results that we have)
plot_df = bias_summary.copy()
severity_map = {'LOW': 1, 'MODERATE': 2, 'HIGH': 3}
plot_df['Severity Score'] = plot_df['Severity'].map(severity_map)

fig = px.bar(
    plot_df,
    x='Bias Type',
    y='Severity Score',
    color='Severity',
    text='Metric',
    hover_data=['Note'],
    color_discrete_map={'LOW': '#6BCB77', 'MODERATE': '#FFD93D', 'HIGH': '#FF6B6B'},
    title='Bias Summary: Severity by Bias Type'
)
fig.update_traces(textposition='outside')
fig.update_layout(
    height=450,
    width=800,
    xaxis_title='Bias Type',
    yaxis_title='Severity (1=Low, 2=Moderate, 3=High)',
    yaxis=dict(tickmode='array', tickvals=[1, 2, 3], ticktext=['Low', 'Moderate', 'High']),
    xaxis_tickangle=-30
)
fig.show()

*Quick interpretation :*
- **Gender bias** *is relatively mild (the dataset is slightly male heavy but finding rates are similar across genders.)*
- **Age bias** *is the biggest concern, older patients have significantly more findings, and age groups are unevenly represented.*
- **Pathology imbalance** *is extreme some conditions appear hundreds of times more than others.*
*(As said previously, these biases could cause a model to perform unfairly: under-diagnosing younger patients, or being less accurate for rare conditions.)*

### 6. **Bias Mitigation: Comparing Pre-processing Techniques**

*Since we can't train a model (no access to images), we apply pre-processing mitigation techniques seen in class that modify the dataset before any model training. We compare three approaches:*

1. **Reweighting** : *assigns sample weights to balance under-represented subgroups without changing the data itself.*
2. **Massaging (Label Flipping)** : *changes labels of borderline samples near the decision boundary to reduce statistical parity difference.*
3. **Uniform Sampling** : *resamples the dataset so that each sensitive subgroup has equal representation.*

*We evaluate each technique on both **gender** and **age** fairness using the same metrics, then compare them side by side.*

In [54]:
# baseline fairness metrics (no mitigation applied), for future comparaisons
baseline_gender = get_metrics(
    y_true=df['has_finding'], y_pred=df['has_finding'],
    prot_attr=df['gender'], priv_group=1, pos_label=1
)

In [55]:
print("Gender (privileged = Male):")
for k, v in baseline_gender.items():
    if v is not None: print(f"  {k}: {v:.4f}")

Gender (privileged = Male):
  base_rate_truth: 0.4682
  statistical_parity_difference: -0.0138
  disparate_impact_ratio: 0.9709
  base_rate_preds: 0.4682
  equal_opportunity_difference: 0.0000
  average_odds_difference: 0.0000
  conditional_demographic_disparity: -0.0010
  smoothed_edf: 0.0296
  df_bias_amplification: 0.0000
  balanced_accuracy_score: 1.0000


In [56]:
baseline_age = get_metrics(
    y_true=df['has_finding'], y_pred=df['has_finding'],
    prot_attr=df['age_binary'], priv_group=1, pos_label=1
)
print("\nAge (privileged = ≥50):")
for k, v in baseline_age.items():
    if v is not None: print(f"  {k}: {v:.4f}")


Age (privileged = ≥50):
  base_rate_truth: 0.4682
  statistical_parity_difference: -0.1693
  disparate_impact_ratio: 0.6975
  base_rate_preds: 0.4682
  equal_opportunity_difference: 0.0000
  average_odds_difference: 0.0000
  conditional_demographic_disparity: 0.0136
  smoothed_edf: 0.3602
  df_bias_amplification: 0.0000
  balanced_accuracy_score: 1.0000


In [58]:

print(f"\nGender SPD = {baseline_gender['statistical_parity_difference']:.4f}, DI = {baseline_gender['disparate_impact_ratio']:.4f}")
print(f"Age SPD = {baseline_age['statistical_parity_difference']:.4f}, DI = {baseline_age['disparate_impact_ratio']:.4f}")


Gender SPD = -0.0138, DI = 0.9709
Age SPD = -0.1693, DI = 0.6975


*As a reminder: gender fairness is already close to ideal (SPD ≈ -0.01, DI ≈ 0.97), while age fairness is clearly problematic (SPD ≈ -0.17, DI ≈ 0.70 which is below the 0.8 threshold). The mitigation techniques below aim to improve these metrics.*

#### 6.1 Reweighting

*Reweighting assigns a sample weight to each observation based on its subgroup. We use `aif360.algorithms.preprocessing.Reweighing`. It computes weights using the formula:*

$$w(g,y) = \frac{P(G=g) \cdot P(Y=y)}{P(G=g, Y=y)}$$

*This accounts for both group membership AND label value, unlike a simple inverse-frequency approach. Under-represented (group, label) combinations get higher weights.*

In [91]:
# convert our dataframe into aif360 StandardDatasets
aif_df_gender = df[['gender', 'has_finding']].copy().astype(float)
dataset_gender = StandardDataset(
    df=aif_df_gender, label_name='has_finding', favorable_classes=[1],
    protected_attribute_names=['gender'], privileged_classes=[[1]]
)

aif_df_age = df[['age_binary', 'has_finding']].copy().astype(float)
dataset_age = StandardDataset(
    df=aif_df_age, label_name='has_finding', favorable_classes=[1],
    protected_attribute_names=['age_binary'], privileged_classes=[[1]]
)

In [92]:
# fit and transform
RW_gender = AIF_Reweighing(unprivileged_groups=[{'gender': 0}], privileged_groups=[{'gender': 1}])
RW_gender.fit(dataset_gender)
dataset_rw_gender = RW_gender.transform(dataset_gender)

RW_age = AIF_Reweighing(unprivileged_groups=[{'age_binary': 0}], privileged_groups=[{'age_binary': 1}])
RW_age.fit(dataset_age)
dataset_rw_age = RW_age.transform(dataset_age)

In [93]:
# store weights back in the dataframe
df['rw_weight_gender'] = dataset_rw_gender.instance_weights
df['rw_weight_age'] = dataset_rw_age.instance_weights
df['rw_weight'] = df['rw_weight_gender'] * df['rw_weight_age']

In [94]:
print("Weights per (group, label) — Gender:")
print(df.groupby(['gender', 'has_finding'])['rw_weight_gender'].first())
print("\nWeights per (group, label) — Age:")
print(df.groupby(['age_binary', 'has_finding'])['rw_weight_age'].first())

Weights per (group, label) — Gender:
gender  has_finding
0       0              0.986283
        1              1.016050
1       0              1.012245
        1              0.986446
Name: rw_weight_gender, dtype: float64

Weights per (group, label) — Age:
age_binary  has_finding
0           0              0.872348
            1              1.199345
1           0              1.207818
            1              0.836515
Name: rw_weight_age, dtype: float64


In [95]:
# fairness metrics after reweighting (using per-attribute weights)
rw_gender = get_metrics(
    y_true=df['has_finding'], y_pred=df['has_finding'],
    prot_attr=df['gender'], priv_group=1, pos_label=1,
    sample_weight=df['rw_weight_gender']
)
rw_age = get_metrics(
    y_true=df['has_finding'], y_pred=df['has_finding'],
    prot_attr=df['age_binary'], priv_group=1, pos_label=1,
    sample_weight=df['rw_weight_age']
)

print("After Reweighting ")
print(f"  Gender :SPD: {rw_gender['statistical_parity_difference']:.4f}, DI: {rw_gender['disparate_impact_ratio']:.4f}")
print(f"  Age    :SPD: {rw_age['statistical_parity_difference']:.4f}, DI: {rw_age['disparate_impact_ratio']:.4f}")

After Reweighting 
  Gender :SPD: 0.0000, DI: 1.0000
  Age    :SPD: 0.0000, DI: 1.0000


*Reweighting assigns $w(g,y) = P(G{=}g) \cdot P(Y{=}y) / P(G{=}g, Y{=}y)$ to each sample. Because each (group, label) combination is rebalanced, the weighted base rates become equal across groups, which gives us a perfect SPD ≈ 0 and DI ≈ 1. This corrects the statistical distribution without modifying actual labels or data.*

#### 6.2 Massaging (Label Flipping)

*Massaging is a pre-processing technique that modifies the labels of borderline samples to equalize the base rates across groups. We rank samples using a KNN-based score (average label of neighbors), then:*
- *In the privileged group: flip the least confident positives → negative.*
- *In the unprivileged group: flip the highest-score negatives → positive.*

*This technique is based on the algorithm proposed by **Kamiran & Calders** in ["Data preprocessing techniques for classification without discrimination" (Knowledge and Information Systems, 2012)](https://doi.org/10.1007/s10115-011-0463-8). It is not available in aif360, so the implementation below is our own adaptation using KNN-based ranking.*

In [ ]:
# we define our own function for massaging labels
def massage_labels(df, prot_attr_col, priv_value, label_col, features):
    df_mass = df.copy()

    # score each sample using average neighbor label (KNN ranking)
    X_scaled = StandardScaler().fit_transform(df_mass[features].values)
    knn = NearestNeighbors(n_neighbors=10).fit(X_scaled)
    _, indices = knn.kneighbors(X_scaled)
    df_mass['_score'] = df_mass[label_col].values[indices[:, 1:]].mean(axis=1)

    # compute flips needed to bring each group to the global base rate
    priv = df_mass[prot_attr_col] == priv_value
    global_rate = df_mass[label_col].mean()
    flips_priv = max(0, int(round((df_mass.loc[priv, label_col].mean() - global_rate) * priv.sum())))
    flips_unpriv = max(0, int(round((global_rate - df_mass.loc[~priv, label_col].mean()) * (~priv).sum())))

    # flip lowest-score positives in privileged group
    if flips_priv > 0:
        idx = df_mass.loc[priv & (df_mass[label_col] == 1)].nsmallest(flips_priv, '_score').index
        df_mass.loc[idx, label_col] = 0

    # flip highest-score negatives in unprivileged group
    if flips_unpriv > 0:
        idx = df_mass.loc[~priv & (df_mass[label_col] == 0)].nlargest(flips_unpriv, '_score').index
        df_mass.loc[idx, label_col] = 1

    return df_mass.drop(columns=['_score']), flips_priv + flips_unpriv

In [71]:
# features available for ranking
ranking_features = ['Patient Age', 'OriginalImage[Width', 'Height]',
                    'OriginalImagePixelSpacing[x', 'y]']

print("Applying massaging for gender:")
df_mass_gender, flips_g = massage_labels(df, 'gender', 1, 'has_finding', ranking_features)
print(f"  Flipped {flips_g} labels ({flips_g/len(df)*100:.2f}% of data)")

print("\nApplying massaging for age:")
df_mass_age, flips_a = massage_labels(df, 'age_binary', 1, 'has_finding', ranking_features)
print(f"  Flipped {flips_a} labels ({flips_a/len(df)*100:.2f}% of data)")

Applying massaging for gender:
  Flipped 104 labels (0.69% of data)

Applying massaging for age:
  Flipped 1262 labels (8.41% of data)


*Massaging modifies only a small fraction of the labels, the "borderline" samples closest to the decision boundary according to their KNN score. For gender, very few flips are needed (since the baseline bias is already small). For age, more flips are required to close the larger gap in finding rates.*

In [74]:
# fairness metrics after massaging
# for gender: use massaged-for-gender labels
mass_gender = get_metrics(
    y_true=df_mass_gender['has_finding'], y_pred=df_mass_gender['has_finding'],
    prot_attr=df_mass_gender['gender'], priv_group=1, pos_label=1
)
# for age: use massaged-for-age labels
mass_age = get_metrics(
    y_true=df_mass_age['has_finding'], y_pred=df_mass_age['has_finding'],
    prot_attr=df_mass_age['age_binary'], priv_group=1, pos_label=1
)

In [76]:

print("After Massaging:")
print(f"  Gender :SPD: {mass_gender['statistical_parity_difference']:.4f}, DI: {mass_gender['disparate_impact_ratio']:.4f}")
print(f"  Age    :SPD: {mass_age['statistical_parity_difference']:.4f}, DI: {mass_age['disparate_impact_ratio']:.4f}")

After Massaging:
  Gender :SPD: 0.0001, DI: 1.0002
  Age    :SPD: 0.0001, DI: 1.0002


#### 6.3 Uniform Sampling

*Uniform sampling creates a balanced dataset by **resampling** so that each sensitive subgroup (Gender × Age Group) has the same number of observations. We use oversampling of minority groups to match the majority group size, preserving all original data.*

In [77]:
# create combined sensitive groups for sampling
df['sensitive_group'] = df['Patient Gender'] + '_' + df['age_group'].astype(str)
group_counts = df['sensitive_group'].value_counts().sort_index()

# uniform sampling: oversample each subgroup to match the largest group
target_size = group_counts.max()
sampled_dfs = []

for group, count in group_counts.items():
    group_df = df[df['sensitive_group'] == group]
    if count < target_size:
        sampled = group_df.sample(n=target_size, replace=True, random_state=42)
    else:
        sampled = group_df.copy()
    sampled_dfs.append(sampled)

df_sampled = pd.concat(sampled_dfs, ignore_index=True)

In [78]:
print(f"Original dataset:  {len(df)} samples")
print(f"Sampled dataset:   {len(df_sampled)} samples")
print(f"\nSubgroup counts after sampling:")
print(df_sampled['sensitive_group'].value_counts().sort_index())
print(f"\nGender split: {df_sampled['Patient Gender'].value_counts().to_dict()}")
print(f"Age binary split: {df_sampled['age_binary'].value_counts().to_dict()}")

Original dataset:  14998 samples
Sampled dataset:   25248 samples

Subgroup counts after sampling:
sensitive_group
F_30-50    3156
F_50-70    3156
F_70+      3156
F_<30      3156
M_30-50    3156
M_50-70    3156
M_70+      3156
M_<30      3156
Name: count, dtype: int64

Gender split: {'F': 12624, 'M': 12624}
Age binary split: {1: 13014, 0: 12234}


In [80]:
# fairness metrics after uniform sampling
samp_gender = get_metrics(
    y_true=df_sampled['has_finding'], y_pred=df_sampled['has_finding'],
    prot_attr=df_sampled['gender'], priv_group=1, pos_label=1
)
samp_age = get_metrics(
    y_true=df_sampled['has_finding'], y_pred=df_sampled['has_finding'],
    prot_attr=df_sampled['age_binary'], priv_group=1, pos_label=1
)

print("After Uniform Sampling:")
print(f"  Gender :SPD: {samp_gender['statistical_parity_difference']:.4f}, DI: {samp_gender['disparate_impact_ratio']:.4f}")
print(f"  Age    :SPD: {samp_age['statistical_parity_difference']:.4f}, DI: {samp_age['disparate_impact_ratio']:.4f}")

After Uniform Sampling:
  Gender :SPD: 0.0103, DI: 1.0213
  Age    :SPD: -0.2064, DI: 0.6496


*Uniform sampling equalizes subgroup sizes but preserves each sample's original label, so it only corrects representation imbalance. it cannot close the gap in finding rates between groups. As a result, age bias is pretty much unchanged.*

#### 6.4 Comparison of Mitigation Techniques

*Let's compare all three techniques side by side. A good mitigation should bring SPD closer to 0 and DI closer to 1.*

In [87]:
# build comparison table
rows = []
techniques = {
    'Baseline': {'gender': baseline_gender, 'age': baseline_age},
    'Reweighting': {'gender': rw_gender, 'age': rw_age},
    'Massaging': {'gender': mass_gender, 'age': mass_age},
    'Sampling': {'gender': samp_gender, 'age': samp_age},
}

for tech, metrics in techniques.items():
    rows.append({
        'Technique': tech,
        'Gender SPD': metrics['gender']['statistical_parity_difference'],
        'Gender DI': metrics['gender']['disparate_impact_ratio'],
        'Gender Smoothed EDF': metrics['gender']['smoothed_edf'],
        'Age SPD': metrics['age']['statistical_parity_difference'],
        'Age DI': metrics['age']['disparate_impact_ratio'],
        'Age Smoothed EDF': metrics['age']['smoothed_edf'],
    })

In [88]:
comparison = pd.DataFrame(rows).set_index('Technique')

# display
for col in comparison.columns:
    comparison[col] = comparison[col].map(lambda x: f"{x:.4f}")
print(comparison.to_string())

            Gender SPD Gender DI Gender Smoothed EDF  Age SPD  Age DI Age Smoothed EDF
Technique                                                                             
Baseline       -0.0138    0.9709              0.0296  -0.1693  0.6975           0.3602
Reweighting     0.0000    1.0000              0.0000   0.0000  1.0000           0.0000
Massaging       0.0001    1.0002              0.0002   0.0001  1.0002           0.0002
Sampling        0.0103    1.0213              0.0211  -0.2064  0.6496           0.4313


In [89]:
# determine best technique for each attribute
comparison_numeric = pd.DataFrame(rows).set_index('Technique')

best_gender = comparison_numeric.loc[comparison_numeric.index != 'Baseline', 'Gender SPD'].abs().idxmin()
best_age = comparison_numeric.loc[comparison_numeric.index != 'Baseline', 'Age SPD'].abs().idxmin()

In [90]:
print("BEST TECHNIQUE")
print(f"\nFor Gender fairness:  {best_gender}")
print(f"  SPD: {comparison_numeric.loc[best_gender, 'Gender SPD']:.4f} (baseline: {comparison_numeric.loc['Baseline', 'Gender SPD']:.4f})")
print(f"  DI:  {comparison_numeric.loc[best_gender, 'Gender DI']:.4f} (baseline: {comparison_numeric.loc['Baseline', 'Gender DI']:.4f})")

print(f"\nFor Age fairness:     {best_age}")
print(f"  SPD: {comparison_numeric.loc[best_age, 'Age SPD']:.4f} (baseline: {comparison_numeric.loc['Baseline', 'Age SPD']:.4f})")
print(f"  DI:  {comparison_numeric.loc[best_age, 'Age DI']:.4f} (baseline: {comparison_numeric.loc['Baseline', 'Age DI']:.4f})")

# overall score: average |SPD| across both attributes (lower = fairer)
print("\nOverall ranking (avg |SPD| across gender + age):")
for tech in tech_names:
    avg_spd = (abs(comparison_numeric.loc[tech, 'Gender SPD']) + abs(comparison_numeric.loc[tech, 'Age SPD'])) / 2
    print(f"  {tech:15s}: {avg_spd:.4f}")

BEST TECHNIQUE

For Gender fairness:  Reweighting
  SPD: 0.0000 (baseline: -0.0138)
  DI:  1.0000 (baseline: 0.9709)

For Age fairness:     Reweighting
  SPD: 0.0000 (baseline: -0.1693)
  DI:  1.0000 (baseline: 0.6975)

Overall ranking (avg |SPD| across gender + age):
  Baseline       : 0.0916
  Reweighting    : 0.0000
  Massaging      : 0.0001
  Sampling       : 0.1083


**Interpretation of Mitigation Comparison:**

- **Reweighting** nearly eliminates gender SPD (→ -0.0001) but actually worsens age SPD (from -0.17 to -0.20). This is because reweighting balances subgroup representation (Gender × Age Group), but doesn't directly address the age-finding correlation. Older groups still have higher finding rates even with equal effective counts.
- **Massaging** is better on both attributes — SPD ≈ 0 for both gender and age. By flipping borderline labels, it directly equalizes the base rates between groups. However, it modifies ~8.4% of labels for age, which is alot and could alter the medical ground truth.
- **Uniform Sampling** balances subgroup sizes but doesn't change the finding rates within each group. So age bias remains strong (SPD ≈ -0.21), similar to reweighting. Gender gets slightly worse due to oversampled groups having different finding rates.

*Reweighting and sampling address **representation bias** (unequal group sizes), but **not label bias** (different finding rates per group). Only massaging addresses label bias directly but at the cost of changing ground truth. For medical data, I believe this must be carefully considered with domain experts.*

In [96]:
# save the mitigated datasets for future model training
# 1. reweighted version (original data + sample weights)
df_export = df.copy()
df_export.to_csv('AL_MASRI_REINA_reweighted.csv', index=False)
print(f"Saved reweighted dataset: AL_MASRI_REINA_reweighted.csv ({df_export.shape})")

# 2. massaged version (labels flipped for gender + age separately — use gender version as default)
df_mass_gender.to_csv('AL_MASRI_REINA_massaged.csv', index=False)
print(f"Saved massaged dataset:   AL_MASRI_REINA_massaged.csv ({df_mass_gender.shape})")

# 3. sampled version
df_sampled.to_csv('AL_MASRI_REINA_sampled.csv', index=False)
print(f"Saved sampled dataset:    AL_MASRI_REINA_sampled.csv ({df_sampled.shape})")

Saved reweighted dataset: AL_MASRI_REINA_reweighted.csv ((14998, 20))
Saved massaged dataset:   AL_MASRI_REINA_massaged.csv ((14998, 19))
Saved sampled dataset:    AL_MASRI_REINA_sampled.csv ((25248, 20))


### 7. Conclusion

**Summary : What we found:**
- The dataset contains **54,490 chest X-ray records** from **14,998 unique patients** (after deduplication).
- **Gender bias is mild**; males are slightly over-represented (53.5% vs 46.5%), but finding rates are similar (~1.4% difference). SPD = -0.014, DI = 0.97 which well above the 0.8 threshold.
- **Age bias is significant**; older patients (≥50) have ~17% higher finding rates than younger patients. SPD = -0.169, DI = 0.70 which is below the 0.8 threshold.
- **Pathology distribution is highly imbalanced**; Infiltration is 46× more frequent than Hernia.
- **Crossed analyses** confirm that age groups are unevenly distributed within genders, and certain pathologies strongly correlate with both age and gender.

**Mitigation comparison of the 3 pre-processing techniques:**
| Technique | Gender SPD | Gender DI | Age SPD | Age DI | Best for |
|-----------|-----------|-----------|---------|--------|----------|
| Baseline | -0.014 | 0.97 | -0.169 | 0.70 | — |
| Reweighting | **-0.0001** | **1.00** | -0.202 | 0.65 | Gender |
| Massaging | **0.0001** | **1.00** | **0.0001** | **1.00** | Both ✓ |
| Sampling | 0.010 | 1.02 | -0.206 | 0.65 | Neither |

- **Massaging** achieves near-perfect fairness on both attributes (SPD ≈ 0, DI ≈ 1) by flipping ~8.4% of age labels and ~0.7% of gender labels.
- **Reweighting** is excellent for gender but ineffective for age, it balances group sizes, not base rates.
- **Uniform Sampling** adds ~10K duplicate samples but doesn't improve fairness on either axis.

**Limitations:**
- We only analyzed **metadata** without the actual images.
- Massaging modifies ground truth labels, which is controversial for medical data.
- Uniform sampling can cause overfitting due to duplicate samples.
- Other sensitive attributes (ethnicity, socioeconomic status) are not available in this dataset.

**For future model training:**
1. For **gender fairness**: use reweighting (`sample_weight` in sklearn), it's effective and preserves labels.
2. For **age fairness**: maybe use massaging, or combine with post-processing techniques.

#### *To be continued..*